In [ ]:
# Step 1: Woody vegetation change detection using DEA Land Cover

In [ ]:
# ===============================================================
# Woody Plant Encroachment using DEA Land Cover (Level-4)
# Windows: fixed cadence (e.g., 5-yr) + custom hydrological epochs
# Period: 1988–2023
# Output: CSV summary (no rasters)
# ===============================================================

import os
import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
import rioxarray
from datacube.utils.geometry import Geometry, CRS
from shapely.geometry import mapping
import datacube
from datetime import datetime

# -------------------
# CONFIG
# -------------------
START_YEAR = 1988
END_YEAR = 2023

# Main cadence (keep 5 for primary; you can re-run with 10 for decadal)
INTERVAL_YRS = 5

# Hydrological epochs to add (edit as needed)
CUSTOM_EPOCHS = [
    {"label": "1988-1993_WetBaseline", "start": 1988, "end": 1993},
    {"label": "1994-2000_EarlyDry", "start": 1994, "end": 2000},
    {"label": "2001-2009_MillenniumDrought", "start": 2001, "end": 2009},
    {"label": "2010-2012_PostDroughtFloods", "start": 2010, "end": 2012},
    {"label": "2013-2017_ModerateDry", "start": 2013, "end": 2017},
    {"label": "2018-2019_SevereDrought", "start": 2018, "end": 2019},
    {"label": "2020-2022_LaNinaFloods", "start": 2020, "end": 2022},
    {"label": "2023_CurrentNeutral", "start": 2023, "end": 2023},
]

PIXEL_RES = 30
AREA_PER_PX_HA = (PIXEL_RES ** 2) / 1e4
TARGET_CRS = "EPSG:3577"
LC_PRODUCT = "ga_ls_landcover_class_cyear_3"
BUFFER_M = 500

TABLE_DIR = "/home/jovyan/DEA products paper/Results/step 1_v6/tables"
os.makedirs(TABLE_DIR, exist_ok=True)

# -------------------
# PATHS: wetlands (5 shapefiles, one per sub-region)
# -------------------
BASE_DIR = "/home/jovyan/DEA products paper/Data"
WETLAND_SHP_PATHS = [
    os.path.join(BASE_DIR, "wetland boundaries/P1ANAEWetlands.shp"),
    os.path.join(BASE_DIR, "wetland boundaries/P2ANAEWetlands.shp"),
    os.path.join(BASE_DIR, "wetland boundaries/P3ANAEWetlands.shp"),
    os.path.join(BASE_DIR, "wetland boundaries/P4ANAEWetlands.shp"),
    os.path.join(BASE_DIR, "wetland boundaries/P5ANAEWetlands.shp"),
]

# -------------------
# CLASS CODE GROUPS (Natural-only + ambiguous NAT kept)
# -------------------
# Natural woody targets
WOODY_CODES = {
    20, 27, 28, 29, 30, 31,
    56, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77
}

# Natural herbaceous sources
HERB_CODES = {
    21, 32, 33, 34, 35, 36,
    57, 78, 79, 80, 81, 82, 83,
    84, 85, 86, 87, 88, 89, 90, 91, 92
}

# Bare & Water (natural)
BARE_CODES = {94, 95, 96, 97}
WATER_CODES = {98, 99, 100, 101, 102, 103, 104}

# Ambiguous NATURAL — included in analysis:
#   source for encroachment (Ambig -> Woody)
#   supplementary retreat target (Woody -> Ambig) reported separately
AMBIG_NAT = {19, 22, 23, 24, 25, 26, 55, 58, 59, 60, 61, 62}

# Exclusions
ARTIFICIAL_CODES = {93}
NODATA_CODES = {255}
CULTIVATED = {1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18}

# Valid pixels exclude cultivated, artificial, NoData (keep ambiguous NAT)
MASK_OUT_CODES = ARTIFICIAL_CODES | NODATA_CODES | CULTIVATED

# -------------------
# HELPERS
# -------------------
def make_periods(start, end, interval_years=None, custom_epochs=None):
    """
    Returns list of (label, y0, y1).

    - interval_years: rolling baseline->target windows (e.g., 5 or 10)
    - custom_epochs: list of dicts {"label": str, "start": int, "end": int}
    """
    periods = []

    if interval_years:
        years = list(range(start, end, interval_years))
        for y0 in years:
            y1 = y0 + interval_years
            if y1 <= end:
                periods.append((f"{y0}-{y1}", y0, y1))

    if custom_epochs:
        for ep in custom_epochs:
            label = ep["label"]
            y0, y1 = int(ep["start"]), int(ep["end"])
            if y0 < y1 and (start <= y0 < y1 <= end):
                periods.append((label, y0, y1))

    return periods


def dc_query_from_geom(geom, crs_str=TARGET_CRS, res=PIXEL_RES, buffer_m=BUFFER_M):
    geom_buf = geom.buffer(buffer_m)
    return {
        "geopolygon": Geometry(mapping(geom_buf), CRS(crs_str)),
        "output_crs": crs_str,
        "resolution": (-res, res),
    }


def mask_codes(da: xr.DataArray, keep: set) -> xr.DataArray:
    codes = np.array(list(keep), dtype="int32")
    return xr.apply_ufunc(
        np.isin,
        da.astype("int32"),
        codes,
        dask="parallelized",
        output_dtypes=[bool]
    )


def count_px(mask: xr.DataArray) -> int:
    return int(mask.sum().values)


def ha(mask: xr.DataArray) -> float:
    return float(mask.sum().values) * AREA_PER_PX_HA


def pct(part, whole):
    return (part / whole * 100.0) if (whole and whole > 0) else np.nan


def get_year_slice(da: xr.DataArray, year: int):
    if "time" not in da.dims:
        return None

    subset = da.sel(time=da.time.dt.year == year)
    if subset.sizes.get("time", 0) == 0:
        return None

    return subset.squeeze(drop=True)


# -------------------
# LOAD WETLANDS
# -------------------
wetland_list = []

for sub_idx, path in enumerate(WETLAND_SHP_PATHS, start=1):
    gdf = gpd.read_file(path).to_crs(TARGET_CRS)

    if "OBJECTID" not in gdf.columns:
        raise ValueError(f"{path} has no OBJECTID field!")

    gdf["WetlandID"] = gdf["OBJECTID"].astype(int)
    gdf["SubRegion"] = f"SR{sub_idx}"
    gdf["Area_ha"] = gdf.geometry.area / 10000.0
    wetland_list.append(gdf)

wetlands = gpd.GeoDataFrame(pd.concat(wetland_list, ignore_index=True), crs=TARGET_CRS)
print("Wetlands loaded:", len(wetlands))

# -------------------
# MAIN
# -------------------
dc = datacube.Datacube(app="WPE_LC_L4_Multiplex")
rows = []

all_periods = make_periods(
    START_YEAR,
    END_YEAR,
    interval_years=INTERVAL_YRS,
    custom_epochs=CUSTOM_EPOCHS
)
print("Total periods (fixed + epochs):", len(all_periods))

current_subregion = None

for _, wr in wetlands.iterrows():
    wid = int(wr["WetlandID"])
    subr = wr["SubRegion"]
    geom = wr.geometry
    area_ha = float(wr["Area_ha"])

    if subr != current_subregion:
        print(f"▶ Processing {subr} ...")
        current_subregion = subr

    # Load full time span once per wetland
    query = dc_query_from_geom(geom)
    ds = dc.load(
        product=LC_PRODUCT,
        measurements=["full_classification"],
        time=(f"{START_YEAR}-01-01", f"{END_YEAR}-12-31"),
        **query
    )

    if ds.sizes.get("time", 0) == 0:
        continue

    da_stack = ds["full_classification"]

    # Clip to wetland
    try:
        da_stack = da_stack.rio.clip([geom], crs=TARGET_CRS, drop=True)
    except Exception:
        print(f"⚠️ Wetland {wid} ({subr}) skipped (no overlap).")
        continue

    for label, y0, y1 in all_periods:
        base = get_year_slice(da_stack, y0)
        tgt = get_year_slice(da_stack, y1)

        if base is None or tgt is None:
            continue

        # Valid pixels: exclude cultivated, artificial, NoData (keep ambiguous natural)
        valid_base = ~mask_codes(base, MASK_OUT_CODES)
        valid_tgt = ~mask_codes(tgt, MASK_OUT_CODES)
        valid = valid_base & valid_tgt

        total_px_clip = base.size
        valid_px_clip = count_px(valid)
        valid_area_ha = valid_px_clip * AREA_PER_PX_HA

        # Group masks
        base_woody = mask_codes(base, WOODY_CODES) & valid
        tgt_woody = mask_codes(tgt, WOODY_CODES) & valid

        base_herb = mask_codes(base, HERB_CODES) & valid
        base_bare = mask_codes(base, BARE_CODES) & valid
        base_water = mask_codes(base, WATER_CODES) & valid
        base_ambig = mask_codes(base, AMBIG_NAT) & valid

        # Encroachment
        enc_herb = base_herb & tgt_woody
        enc_bare = base_bare & tgt_woody
        enc_water = base_water & tgt_woody
        enc_ambig = base_ambig & tgt_woody
        enc_any = enc_herb | enc_bare | enc_water | enc_ambig

        # Retreat (clear)
        ret_herb = base_woody & mask_codes(tgt, HERB_CODES) & valid
        ret_bare = base_woody & mask_codes(tgt, BARE_CODES) & valid
        ret_water = base_woody & mask_codes(tgt, WATER_CODES) & valid
        ret_any_clear = ret_herb | ret_bare | ret_water

        # Retreat (supplementary: woody -> ambiguous NAT)
        ret_ambig = base_woody & mask_codes(tgt, AMBIG_NAT) & valid
        ret_any_ext = ret_any_clear | ret_ambig

        # Stability bookkeeping (optional)
        base_nonwoody = (~base_woody) & valid
        tgt_nonwoody = (~tgt_woody) & valid

        nw_to_nw = base_nonwoody & tgt_nonwoody
        nw_to_w = base_nonwoody & tgt_woody
        w_to_nw = base_woody & tgt_nonwoody
        w_to_w = base_woody & tgt_woody

        # Areas (ha)
        trees_base_ha = ha(base_woody)
        trees_tgt_ha = ha(tgt_woody)

        enc_herb_ha = ha(enc_herb)
        enc_bare_ha = ha(enc_bare)
        enc_water_ha = ha(enc_water)
        enc_ambig_ha = ha(enc_ambig)
        enc_any_ha = ha(enc_any)

        ret_h2h_ha = ha(ret_herb)
        ret_w2b_ha = ha(ret_bare)
        ret_w2w_ha = ha(ret_water)
        ret_any_clear_ha = ha(ret_any_clear)

        ret_w2a_ha = ha(ret_ambig)
        ret_any_ext_ha = ha(ret_any_ext)

        d_trees_ha = trees_tgt_ha - trees_base_ha
        pct_trees = pct(d_trees_ha, trees_base_ha)

        years_len = max(1, (y1 - y0))  # avoid divide-by-zero for single-year epochs

        # Per-year rates (ha/yr) to compare unequal durations
        enc_any_ha_peryr = enc_any_ha / years_len
        ret_clear_ha_peryr = ret_any_clear_ha / years_len
        ret_extended_ha_peryr = ret_any_ext_ha / years_len
        d_trees_ha_peryr = d_trees_ha / years_len

        rows.append({
            "WetlandID": wid,
            "SubRegion": subr,
            "Period": label,
            "BaselineYear": y0,
            "TargetYear": y1,
            "YearsLen": years_len,
            "Area_ha": area_ha,
            "Valid_px": valid_px_clip,
            "Total_px_clip": total_px_clip,
            "Valid_area_ha": valid_area_ha,

            "Trees_Base_ha": trees_base_ha,
            "Trees_Target_ha": trees_tgt_ha,
            "ΔTrees_ha": d_trees_ha,
            "%ΔTrees": pct_trees,
            "ΔTrees_ha_peryr": d_trees_ha_peryr,

            # Encroachment
            "Enc_Herb_to_Woody_ha": enc_herb_ha,
            "Enc_Bare_to_Woody_ha": enc_bare_ha,
            "Enc_Water_to_Woody_ha": enc_water_ha,
            "Enc_Ambig_to_Woody_ha": enc_ambig_ha,
            "Enc_Any_to_Woody_ha": enc_any_ha,
            "Enc_Any_to_Woody_ha_peryr": enc_any_ha_peryr,

            # Retreat (clear + supplementary)
            "Ret_Woody_to_Herb_ha": ret_h2h_ha,
            "Ret_Woody_to_Bare_ha": ret_w2b_ha,
            "Ret_Woody_to_Water_ha": ret_w2w_ha,
            "Ret_Woody_to_Ambig_ha": ret_w2a_ha,
            "Ret_Any_to_NonWoody_ha": ret_any_clear_ha,
            "Ret_Any_to_NonWoody_Extended_ha": ret_any_ext_ha,
            "Ret_Any_to_NonWoody_ha_peryr": ret_clear_ha_peryr,
            "Ret_Any_to_NonWoody_Extended_ha_peryr": ret_extended_ha_peryr,

            # Transition bookkeeping
            "NW_to_NW_px": count_px(nw_to_nw),
            "NW_to_W_px": count_px(nw_to_w),
            "W_to_NW_px": count_px(w_to_nw),
            "W_to_W_px": count_px(w_to_w),
            "NW_Base_px": count_px(base_nonwoody),
            "W_Base_px": count_px(base_woody),
        })

# -------------------
# EXPORT
# -------------------
summary = pd.DataFrame(rows).sort_values(
    ["SubRegion", "WetlandID", "BaselineYear", "TargetYear"]
).reset_index(drop=True)

ts = datetime.now().strftime("%Y%m%d_%H%M%S")
csv_name = f"WPE_summary_{START_YEAR}_{END_YEAR}_{INTERVAL_YRS}yr_plusEpochs_{ts}.csv"
csv_path = os.path.join(TABLE_DIR, csv_name)

summary.to_csv(csv_path, index=False)

print("✅ Exported summary to:", csv_path)
summary.head()